# Evaluación Parcial N.° 1: Clasificación Curricular Automática con MLP

**Integrantes:**
* Rahyzza Novoa
* Nicolas Salas

**Asignatura:**
* DEEP LEARNING_011V_OLS

## 1. Importaciones y Fijación de Semillas

In [ ]:
import os
import random
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.optimizers import Adam, SGD, RMSprop
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow Version:", tf.__version__)
print("Semillas fijadas en:", SEED)

TensorFlow Version: 2.21.0
Semillas fijadas en: 42


## 2. Carga, Revisión y Distribución del Dataset

In [ ]:
# 1. Cargar el dataset
df = pd.read_csv('Dataset_Clasificacion_Curricular_100000_LIMPIO.csv')

print(f"Dimensiones del dataset: {df.shape}")
print("\nPrimeras filas:")
display(df.head(3))

target_col = df.columns[-1]
print(f"\nVariable objetivo identificada: '{target_col}'")
print("\nDistribución de clases:")
print(df[target_col].value_counts(normalize=True) * 100)

X = df.drop(columns=[target_col]).values
y = df[target_col].values

num_classes = len(np.unique(y))
num_features = X.shape[1]
print(f"\nNúmero de características (features): {num_features}")
print(f"Número de clases: {num_classes}")

# 2. Split estratificado: 70% Train, 15% Validation, 15% Test
# Primer corte: 70% Train, 30% Temporal (Val + Test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)

# Segundo corte sobre el 30% temporal: 50% Val (15% total), 50% Test (15% total)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

print(f"\nTamaño conjunto de entrenamiento (Train): {X_train.shape[0]} muestras")
print(f"Tamaño conjunto de validación (Val):      {X_val.shape[0]} muestras")
print(f"Tamaño conjunto de prueba (Test):          {X_test.shape[0]} muestras")

Dimensiones del dataset: (100000, 23)

Primeras filas:


,nivel_escolar,duracion_estimada_min,numero_paginas,numero_items,porcentaje_visual,densidad_simbolos,prop_operaciones_aritmeticas,prop_fracciones_decimales,prop_variables_algebraicas,prop_ecuaciones,...,prop_coordenadas,prop_tablas_graficos,prop_estadistica_descriptiva,prop_probabilidad,prop_contexto_problemas,dificultad_estimada,ejemplos_resueltos,elementos_visuales,indice_interactividad,categoria_curricular
0,7,41,18,25,54.42,0.2335,0.4200,0.6755,0.0752,0.1390,...,0.1448,0.5649,0.6283,0.2008,0.4641,2,0,5,0.3486,0
1,6,62,19,28,31.81,0.5357,0.5404,0.5512,0.4638,0.6847,...,0.3178,0.2299,0.2365,0.1484,0.7197,3,4,5,0.3725,1
2,12,49,12,27,46.90,0.3354,0.1622,0.3779,0.2806,0.4657,...,0.3686,0.0967,0.1067,0.0050,0.3689,3,2,6,0.2390,2



Variable objetivo identificada: 'categoria_curricular'

Distribución de clases:
categoria_curricular
0    30.219
1    26.967
2    23.065
3    19.749
Name: proportion, dtype: float64

Número de características (features): 22
Número de clases: 4

Tamaño conjunto de entrenamiento (Train): 70000 muestras
Tamaño conjunto de validación (Val):      15000 muestras
Tamaño conjunto de prueba (Test):          15000 muestras
